# Nonlinear management optimization with PESTPP-SQP

In the [PESTPP-OPT notebooks](../part2_08_opt/freyberg_opt_1.ipynb) we saw how to jam the worlds of history matching and management optimization together into a single, formal, constrained optimization problem. PESTPP-OPT solves that problem with _sequential linear programming_ (SLP): at each iteration it builds a linear "response matrix" that maps decision variables to constraints, then hands that linear problem to a simplex solver. That linearity assumption is what makes PESTPP-OPT fast and mature - but it is also its Achilles heel. If the relation between your decision variables (e.g. pumping rates) and your constraints (e.g. surface-water/groundwater exchange) is meaningfully _nonlinear_, the linear response matrix is only a local approximation and the SLP solution can wander.

Enter `PESTPP-SQP`.

`PESTPP-SQP` solves the same kind of constrained management optimization problem

$$\text{maximize (or minimize)} \quad f(x)$$
$$\text{subject to} \quad g_i(x) \le 0, \quad h_j(x) = 0$$

where $x$ is our vector of decision variables, $f$ is the management objective, and the $g$ and $h$ are model-based constraints. But instead of linear programming, it uses _sequential quadratic programming_ (SQP): at each iteration it builds a local _quadratic_ model of the problem and takes a step towards the optimum, respecting the (possibly nonlinear) constraints along the way. This lets `PESTPP-SQP` handle genuinely nonlinear decision-variable/constraint relations that would trip up `PESTPP-OPT`.

There is one more trick up its sleeve, and it is a big one. Rather than computing the gradients it needs with expensive finite differences (one model run per decision variable, per iteration - ouch!), `PESTPP-SQP` can estimate those gradients from an _ensemble_ of decision-variable realizations, using the "Stochastic Simplex Approximate Gradient" (StoSAG) approach. This is the same ensemble magic that makes `PESTPP-IES` so scalable: the cost of a gradient becomes (roughly) independent of the number of decision variables. __You choose the ensemble size__; more realizations buy you better gradients at the cost of more model runs per iteration.

The great news is that we do not have to build the optimization problem from scratch. The [PESTPP-OPT notebook](../part2_08_opt/freyberg_opt_1.ipynb) already did all the hard work - defining the decision variables, the management objective, and the constraints - and left it all sitting in its template folder. So in this notebook we simply _reuse_ that setup and swap the solver: instead of running `PESTPP-OPT`, we run `PESTPP-SQP`. Same "how much water can we pump without harming the stream?" problem, solved with ensemble-based SQP.

In the [next notebook](freyberg_sqp_2.ipynb) we bring parameter uncertainty into the mix and solve the same problem under uncertainty using an ensemble "stack".

> __Note__: `PESTPP-SQP` is the newest of the PEST++ constrained-optimization tools. It is powerful but has more moving parts than PESTPP-OPT, so it pays to inspect the run record and to sanity-check its answers against a simpler formulation where you can.

### Admin

Start off with the usual loading of dependencies. Simply run the next cell by pressing `shift+enter`.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import pandas as pd
font = {'size'   : 10}
import matplotlib
matplotlib.rc('font', **font)
import matplotlib.pyplot as plt;
import shutil
import psutil

import sys
import pyemu
import flopy
assert "dependencies" in flopy.__file__
assert "dependencies" in pyemu.__file__
sys.path.insert(0,"..")
import herebedragons as hbd

### Reuse the PESTPP-OPT problem setup

Rather than rebuild the whole management-optimization problem, we lean on the work already done in the ["freyberg opt 1"](../part2_08_opt/freyberg_opt_1.ipynb) notebook. That notebook prepared a PEST interface with:

- the future well-flux multipliers set up as _decision variables_ (parameter group `decvars`),
- a management _objective function_ (`obj_well`) that maximizes total future groundwater extraction, and
- the surface-water/groundwater exchange and minimum-supply _constraints_ (`less_than_swgw` and `less_than_wel`).

It left all of that in its template folder, so we just copy it across. Run the next cell - it will complain if you have not run the PESTPP-OPT notebook yet.

In [ ]:
# specify the temporary working folder
t_d = os.path.join('freyberg6_template')
if os.path.exists(t_d):
    shutil.rmtree(t_d)

# the pestpp-opt template, which already has the decision variables, objective and constraints set up
org_t_d = os.path.join("..","part2_08_opt","freyberg6_template")
assert os.path.exists(org_t_d), "you need to run the '/part2_08_opt/freyberg_opt_1.ipynb' notebook first"

shutil.copytree(org_t_d,t_d)
pst_path = os.path.join(t_d, 'pest.pst')

Load the control file and take a quick look at what the PESTPP-OPT notebook set up for us.

In [ ]:
pst = pyemu.Pst(pst_path)

In [ ]:
# the decision variables and the management objective / constraints are already in place:
par = pst.parameter_data
obs = pst.observation_data
dv_names = par.loc[par.pargp=="decvars","parnme"]
print("number of decision variables:",dv_names.shape[0])
print("optimization direction:     ",pst.pestpp_options["opt_direction"])
print("objective function:         ",pst.pestpp_options["opt_objective_function"])
print("decision-variable group:    ",pst.pestpp_options["opt_dec_var_groups"])
print("constraint groups:          ",[g for g in pst.nnz_obs_groups])

### From SLP to ensemble SQP: what actually changes

Reusing the PESTPP-OPT setup means the _problem_ is identical. All we need to change is the _solver_. There are just two things to take care of.

__1. A feasible starting point.__ `PESTPP-OPT` happily started every decision variable pinned at its lower bound (a multiplier of 0.0, i.e. all wells off). `PESTPP-SQP` in ensemble mode is different: it draws its initial decision-variable ensemble _around_ the starting values, so we want to start from a sensible, _feasible_ interior point rather than at a bound. Here we start every future well at a multiplier of 1.0 - "business as usual" current pumping - which (as we will confirm below) satisfies all the constraints. The ensemble then spreads out around that point and StoSAG uses it to sense which way is "uphill".

In [ ]:
par.loc[dv_names,"parval1"] = 1.0  # feasible starting point (current pumping)

__2. Turn on ensemble gradients and choose the ensemble size.__ This is the key `PESTPP-SQP` control. Setting `sqp_num_reals` greater than zero switches on _ensemble-based gradients_ (StoSAG): at each iteration `PESTPP-SQP` evaluates an ensemble of decision-variable realizations and uses the statistical relation between decision-variable perturbations and objective/constraint responses to estimate the gradients it needs. (Setting it to zero would fall back to finite-difference gradients - one run per decision variable - which for our 84 decision variables would be much more expensive.)

More realizations generally give better gradient estimates, but each realization is a model run _per iteration_, so the cost scales with the ensemble size. Pick a value appropriate for your problem size and available compute - a few tens of realizations is a reasonable place to start.

> __Go ahead and change `num_reals`__ to suit your machine and your patience!

In [ ]:
num_reals = 30 # the decision-variable ensemble size for StoSAG gradients - choose to suit your resources!
pst.pestpp_options["sqp_num_reals"] = num_reals

### Run PESTPP-SQP

As always, let's first check that the PEST setup is valid and runs with a `noptmax=0` run.

In [ ]:
pst.control_data.noptmax = 0
pst.write(pst_path,version=2)
pyemu.os_utils.run("pestpp-sqp pest.pst",cwd=t_d)

If that ran without error, we are good to go. Now re-write the control file with a non-zero `noptmax`. For `PESTPP-SQP`, `noptmax` is the number of SQP iterations. A handful of iterations is usually enough to see the solution take shape.

> __Heads up__: ensemble SQP does real work each iteration - it runs the decision-variable ensemble _plus_ a line search - so this run takes several minutes. Increase `noptmax` for a more converged solution, or dial back `num_reals` above to go faster.

In [ ]:
pst.control_data.noptmax = 3
pst.write(pst_path,version=2)

# Attention!

You must specify a number of workers which is adequate for ***your*** machine! Make sure to assign an appropriate value for the following `num_workers` variable:

In [ ]:
num_workers = 10 # update according to your available resources!

Specify the folder in which the PEST manager will run and record outcomes. It should be different from the `t_d` folder.

In [ ]:
m_d = os.path.join('master_sqp_1')

The following cell deploys the PEST agents and manager and then starts the run using `pestpp-sqp`. Run it by pressing `shift+enter`.

If you wish to see the outputs in real-time, switch over to the terminal window (the one which you used to launch the `jupyter notebook`). There you should see `pestpp-sqp`'s progress. At each SQP iteration the manager issues the ensemble of decision-variable realizations to the agents, uses their responses to estimate gradients, and takes a step.

In [ ]:
pyemu.os_utils.start_workers(t_d,"pestpp-sqp","pest.pst",num_workers=num_workers,worker_root=".",
                           master_dir=m_d)

### Processing PESTPP-SQP

Because we ran in _ensemble_ mode, the "solution" at each iteration is not a single point but an _ensemble_ of decision-variable realizations that `PESTPP-SQP` marched towards the optimum. So we summarize each iteration by the ensemble _mean_ and look at the _spread_ to see the algorithm's exploration.

`PESTPP-SQP` saves the decision-variable ensemble (`pest.<N>.par.csv`) and the corresponding model-output ensemble (`pest.<N>.obs.csv`) for each iteration `N`. Here's a little helper to grab them in order.

In [ ]:
def iter_ensemble_files(m_d,suffix):
    """list the per-iteration ensemble files (pest.<N><suffix>) sorted by iteration number"""
    found = []
    for f in os.listdir(m_d):
        if f.startswith("pest.") and f.endswith(suffix):
            mid = f[len("pest."):-len(suffix)]
            if mid.isdigit():
                found.append((int(mid),f))
    return [f for _,f in sorted(found)]

par_files = iter_ensemble_files(m_d,".par.csv")
obs_files = iter_ensemble_files(m_d,".obs.csv")
par_files

#### Objective function history

The objective is the total future extraction - the sum of the pumping multipliers (our `obj_well` equation). Let's compute it for every realization at every iteration, and track the ensemble mean (and min/max spread). We asked `PESTPP-SQP` to _maximize_ this, so we hope to see it climb.

In [ ]:
obj_mean,obj_lo,obj_hi = [],[],[]
for f in par_files:
    pe = pd.read_csv(os.path.join(m_d,f),index_col=0)
    obj_real = pe.loc[:,dv_names.values].sum(axis=1) # objective for each realization
    obj_mean.append(obj_real.mean())
    obj_lo.append(obj_real.min())
    obj_hi.append(obj_real.max())

fig,ax = plt.subplots(1,1,figsize=(7,4))
its = np.arange(len(obj_mean))
ax.fill_between(its,obj_lo,obj_hi,alpha=0.2,label="ensemble spread")
ax.plot(its,obj_mean,marker='o',label="ensemble mean")
ax.set_xlabel("SQP iteration")
ax.set_ylabel("objective function\n(total future extraction multiplier)")
ax.set_title("PESTPP-SQP objective history")
ax.legend()
ax.grid()
plt.tight_layout()
plt.show()

The objective climbs as `PESTPP-SQP` marches the ensemble uphill towards more pumping. (If it is still climbing steeply at the last iteration, bump up `noptmax` and re-run to let it converge further.)

#### Was the starting point feasible?

Recall we started the decision variables at a multiplier of 1.0. The iteration-0 ensemble lets us confirm that starting point was _feasible_ - all constraints satisfied. We read the iteration-0 model-output ensemble and check the ensemble-mean constraint values.

In [ ]:
swgw_constraint_names = obs.loc[obs.obgnme=="less_than_swgw","obsnme"].tolist()
wel_constraint_names = obs.loc[obs.obgnme=="less_than_wel","obsnme"].tolist()
swgw_rhs = obs.loc[swgw_constraint_names,"obsval"].max()
wel_rhs = obs.loc[wel_constraint_names,"obsval"].max()

oe0 = pd.read_csv(os.path.join(m_d,obs_files[0]),index_col=0)
print("iteration-0 (starting) ensemble-mean constraint values:")
print("  sw-gw exchange: worst = {0:.0f}, required <= {1:.0f}  -> {2}".format(
      oe0[swgw_constraint_names].mean().max(), swgw_rhs,
      "FEASIBLE" if oe0[swgw_constraint_names].mean().max() <= swgw_rhs else "INFEASIBLE"))
print("  water use:      worst = {0:.0f}, required <= {1:.0f}  -> {2}".format(
      oe0[wel_constraint_names].mean().max(), wel_rhs,
      "FEASIBLE" if oe0[wel_constraint_names].mean().max() <= wel_rhs else "INFEASIBLE"))

#### The optimal solution

The optimal decision variables are the ensemble mean of the _final_ iteration, and the constraint values at that solution are the mean of the final model-output ensemble.

In [ ]:
pe_final = pd.read_csv(os.path.join(m_d,par_files[-1]),index_col=0)
opt = pe_final.loc[:,dv_names.values].mean() # mean optimal multiplier per well/stress-period
oe_final = pd.read_csv(os.path.join(m_d,obs_files[-1]),index_col=0)

Time for some (unavoidable) plotting hackery so we can visualize the pattern of optimal groundwater use across the future stress periods, together with the constraint information. Feel free to skip to the final plot for the message.

In [ ]:
# organise the optimal (mean) decision-variable values by well and stress period
wpar = par.loc[dv_names,:].copy()
wpar["inst"] = wpar.inst.astype(int)
wpar["kij"] = wpar.apply(lambda x: (x.idx0,x.idx1,x.idx2),axis=1)
wpar["optimal"] = opt.loc[wpar.parnme].values
inst_vals = sorted(wpar.inst.unique())
vals = {}
for inst in inst_vals:
    ipar = wpar.loc[wpar.inst==inst,:].copy()
    ipar.sort_values(by="kij",inplace=True)
    ipar.index = ipar.kij
    vals[inst] = ipar.optimal

In [ ]:
# mean constraint values at the optimal solution
swgw_opt = oe_final[swgw_constraint_names].mean()
wel_opt = oe_final[wel_constraint_names].mean()

fig,axes = plt.subplots(2,1,figsize=(12,6))
colors = ["r","g","b","c","m","y","0.5"]
df = pd.DataFrame(vals).T
df.plot(ax=axes[0],kind="bar",color=colors)
axes[0].set_ylim(0,6.5)
axes[0].set_title("optimal (mean) pumping multiplier per well and future stress period")
axes[0].set_ylabel("pumping multiplier")
if axes[0].get_legend() is not None:
    axes[0].get_legend().remove()

axes[1].plot(np.arange(len(wel_constraint_names)),wel_opt.values,"b",lw=1.5,label="sim water use")
axes[1].plot(axes[1].get_xlim(),[wel_rhs,wel_rhs],"b--",lw=2.5,label="water-use constraint")
axt = plt.twinx(axes[1])
axt.plot(np.arange(len(swgw_constraint_names)),swgw_opt.values,"m",lw=1.5,label="sim sw-gw")
axt.plot(axes[1].get_xlim(),[swgw_rhs,swgw_rhs],"m--",lw=2.5,label="sw-gw constraint")
axes[1].set_xlabel("future stress period")
axes[1].set_ylabel("water use ($L^3/T$)",color="b")
axt.set_ylabel("sw-gw exchange ($L^3/T$)",color="m")
axes[1].set_title("constraints at the optimal solution")
plt.tight_layout()
plt.show()

The plot above shows:

 - (top) Optimal (ensemble-mean) pumping multipliers at each well during each future stress period. Wells are distinguished by different coloured bars.
 - (bottom) Total simulated water use (blue) against its minimum-supply constraint (blue dashed), and simulated sw-gw exchange (magenta) against its ecological constraint (magenta dashed).

`PESTPP-SQP` has pushed total extraction up from the "business as usual" starting point, driving the sw-gw exchange towards - but not past - its ecological limit, while still meeting the minimum-supply requirement. Because SQP does not assume a linear response, this solution respects the true (nonlinear) model behaviour at the constraints.

### But what about uncertainty?

Everything we just did was _risk-neutral_: we used a single set of "best-estimate" (calibrated) parameters and trusted the model's constraint predictions completely. But we spent many notebooks learning that model predictions are _uncertain_. What if parameter uncertainty means our "optimal" pumping actually violates the ecological constraint some of the time?

In the [next notebook](freyberg_sqp_2.ipynb) we carry parameter uncertainty into the optimization by feeding `PESTPP-SQP` a "stack" - an ensemble of parameter realizations (conveniently, the posterior ensemble from `PESTPP-IES`) - and solving the same problem under a chosen _risk_ stance. Onward!